In [ ]:
pip install lightgbm

In [ ]:
import xgboost
print(xgboost.__version__)

In [ ]:
import lightgbm
print(lightgbm.__version__)

In [ ]:
import joblib
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)


In [ ]:
X_train = joblib.load("X_train.pkl")
X_test = joblib.load("X_test.pkl")

y_train = joblib.load("y_train.pkl")
y_test = joblib.load("y_test.pkl")

print(X_train.shape)
print(X_test.shape)


In [ ]:
scale_pos_weight = (
    (y_train == 0).sum()
    /
    (y_train == 1).sum()
)

print("Scale Pos Weight :", scale_pos_weight)


In [ ]:
models = {
    "Logistic Regression":
    LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "LightGBM":
    LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        random_state=42,
        class_weight="balanced"
    ),

    "XGBoost":
    XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.2,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    )
}

In [ ]:
results = []
best_f1 = 0
best_model = None
best_name = ""

for name, model in models.items():
    print("="*60)
    print(name)
    print("="*60)

    model.fit(
        X_train, y_train
    )
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:,1]

    accuracy = accuracy_score(y_test,pred)
    precision = precision_score(y_test,pred)
    recall = recall_score(y_test,pred)
    f1 = f1_score(y_test,pred)
    roc = roc_auc_score(y_test,prob)
    pr_auc = average_precision_score(y_test,prob)

    print()
    print(classification_report(y_test,pred))

    print()
    print(confusion_matrix(y_test,pred))

    print()
    print("ROC AUC :", roc)
    print("PR AUC :", pr_auc)

    results.append([name,
        accuracy,
        precision,
        recall,
        f1,
        roc,
        pr_auc])

    if f1 > best_f1:
        best_f1 = f1
        best_model = model
        best_name = name

In [ ]:
joblib.dump(
    best_model,
    "best_model.pkl"
)


In [ ]:
results = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "PR_AUC"]
)
results = results.sort_values(
    by="F1",
    ascending=False
)

print()
print("="*70)
print("FINAL RESULTS")
print("="*70)
print(results)
print()
print("BEST MODEL : ", best_name)
print("BEST F1 :", best_f1)

results.to_csv(
    "model_comparison.csv",
    index=False
)

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
import numpy as np

model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.2,
    scale_pos_weight=6.76,
    eval_metric="logloss",
    random_state=42
)

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

print(scores)
print(np.mean(scores))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("feature_dataset_final.csv")

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

print("Train Unique:",
      train_df["combined_text"].nunique())

print("Test Unique:",
      test_df["combined_text"].nunique())

common = set(
    train_df["combined_text"]
).intersection(
    set(test_df["combined_text"])
)

print("Common Texts :", len(common))

In [ ]:
import pandas as pd

df = pd.read_csv("feature_dataset_final.csv")

print(df["company_name"].nunique())
print(df["job_title"].nunique())

In [ ]:
print(df["source"].value_counts())